## Exercise 1: Build the CSVReader Class (OOP Foundation)

In [1]:
import pandas as pd

In [2]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def read_csv(self):
        self.df = pd.read_csv(self.file_path)
        print(f"CSV file '{self.file_path}' has been read successfully.")
        return self.df
    def preview(self, n):
        if self.df is None:
            self.read_csv()
            
        print(self.df.head(n))
    

In [5]:
# test
reader = CSVReader('sample_data (2).csv')
reader.read_csv()
reader.preview(5)


CSV file 'sample_data (2).csv' has been read successfully.
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Exercise 2: Apply Pattern for Missing Values Handling

In [6]:
from abc import ABC, abstractmethod

In [8]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()
    
class FillMean(MissingValueStrategy):
    def handle(self, df):
        df = df.copy()
        for column in df.select_dtypes(include="number").columns:
            df[column] = df[column].fillna(df[column].mean())
            
        return df
    
class FillMode(MissingValueStrategy):
    def handle(self, df):
        df = df.copy()
        
        for column in df.columns:
            mode_value = df[column].mode()
            
            if not mode_value.empty:
                df[column] = df[column].fillna(mode_value[0])
                
        return df
    
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy
        
    def clean(self, df):
        return self.strategy.handle(df)
    
        

In [11]:
# Test all three strategies
df = pd.read_csv('sample_data (2).csv')

test = {
    "drop": DropMissing(),
    "mean": FillMean(),
    "mode": FillMode()
}

for name, strategy in test.items():
    print(f'\n===== {name} =====')

    cleaner = DataCleaner(strategy)
    cleaned_df = cleaner.clean(df)

    print(cleaned_df)

    print("\nMissing values:")
    print(cleaned_df.isnull().sum())

    print("\nData shape:", cleaned_df.shape)


===== drop =====
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

Missing values:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64

Data shape: (2, 7)

===== mean =====
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

Missing values:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
d


## Exercise 3: Add Decorators for Logging and Timing

In [16]:
import time
import functools

In [ ]:
def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished {func.__name__}")
        return result
    return wrapper


def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"[TIME] {func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper


class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n=5):
        print(self.df.head(n))
        
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        return self.strategy.handle(df)



## Exercise 4: Implement Factory Pattern for Data Transformations

In [20]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    # makes column names lowercase with underscores
    def apply(self, df):
        df = df.copy()
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
        return df


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates()


class StandardizeText(DataTransform):
    # strip spaces + title case for all text columns
    def apply(self, df):
        df = df.copy()
        for col in df.select_dtypes(include=["object", "string"]).columns:
            df[col] = df[col].str.strip().str.title()
        return df


class TransformFactory:
    _transforms = {
        "normalize": NormalizeColumns,
        "duplicates": RemoveDuplicates,
        "text": StandardizeText,
    }

    @staticmethod
    def create(name):
        cls = TransformFactory._transforms.get(name)
        if cls is None:
            raise ValueError(f"Unknown transform: '{name}'. "
                             f"Choose from {list(TransformFactory._transforms)}")
        return cls()

## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

In [ ]:
class DataPipeline(ABC):
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.df

    @abstractmethod
    def load(self): pass

    @abstractmethod
    def clean(self): pass

    @abstractmethod
    def transform(self): pass

    @abstractmethod
    def save(self): pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, strategy, transform_names):
        self.input_path = input_path
        self.output_path = output_path
        self.strategy = strategy
        self.transform_names = transform_names
        self.df = None

    def load(self):
        self.df = CSVReader(self.input_path).read()

    def clean(self):
        self.df = DataCleaner(self.strategy).clean(self.df)

    def transform(self):
        for name in self.transform_names:
            t = TransformFactory.create(name)
            self.df = t.apply(self.df)

    @log_action
    def save(self):
        self.df.to_csv(self.output_path, index=False)


if __name__ == "__main__":
    pipeline = CSVDataPipeline(
        input_path="sample_data (2).csv",
        output_path="cleaned_data.csv",
        strategy=FillMean(),
        transform_names=["normalize", "duplicates", "text"],
    )
    result = pipeline.run()
    print(result)

[LOG] Starting read...
[TIME] read took 0.0083 seconds
[LOG] Finished read
[LOG] Starting save...
[LOG] Finished save
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0
